In [ ]:
!pip -q install pyspark==3.4.4

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.4/311.4 MB 4.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 16.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.4.4 which is incompatible.


In [ ]:
from pyspark.sql import SparkSession
from pyspark.ml.fpm import FPGrowth
import pyspark.sql.functions as F

spark = (
    SparkSession.builder
    .appName("FPGrowth")
    .config(
        "spark.jars.packages",
        "net.snowflake:snowflake-jdbc:3.13.22,"
        "net.snowflake:spark-snowflake_2.12:2.16.0-spark_3.4"
    )
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

In [ ]:
from google.colab import userdata

sf_account = userdata.get("SNOWFLAKE_ACCOUNT")

sfOptions = {
    "sfURL": f"{sf_account}.snowflakecomputing.com",
    "sfUser": userdata.get("SNOWFLAKE_USER"),
    "sfPassword": userdata.get("SNOWFLAKE_PASSWORD"),
    "sfDatabase": "BIGDATA_DB",
    "sfWarehouse": userdata.get("SNOWFLAKE_WAREHOUSE"),
}

sfOptionsStaging = dict(sfOptions)
sfOptionsStaging["sfSchema"] = "STAGING"

sfOptionsRaw = dict(sfOptions)
sfOptionsRaw["sfSchema"] = "RAW"

SNOWFLAKE_SOURCE_NAME = "net.snowflake.spark.snowflake"

In [ ]:
df_user_portfolios = (
    spark.read
    .format(SNOWFLAKE_SOURCE_NAME)
    .options(**sfOptionsStaging)
    .option("dbtable", "USER_PORTFOLIOS")
    .load()
)

df_token_map = (
    spark.read
    .format(SNOWFLAKE_SOURCE_NAME)
    .options(**sfOptionsStaging)
    .option("dbtable", "MAP_TOKEN")
    .load()
)

df_tokens = (
    spark.read
    .format(SNOWFLAKE_SOURCE_NAME)
    .options(**sfOptionsRaw)
    .option("dbtable", "TOKENS")
    .load()
)

In [ ]:
df_portfolio = df_user_portfolios.alias("p").join(
    df_token_map.alias("m"),
    F.col("p.token_id") == F.col("m.token_id"),
    "inner"
).join(
    df_tokens.alias("t"),
    F.lower(F.col("m.token_address")) == F.lower(F.col("t.address")),
    "inner"
).select(
    F.col("p.user_id").alias("user_id"),
    F.col("m.token_address").alias("token_address"),
    F.col("t.symbol").alias("symbol"),
    F.col("p.balance").alias("balance"),
    F.col("p.tx_count").alias("tx_count")
)

df_portfolio.show(5, truncate=False)

+--------+------------------------------------------+------+------------------+--------+
|user_id |token_address                             |symbol|balance           |tx_count|
+--------+------------------------------------------+------+------------------+--------+
|3025051 |0x013bbbf0154b9ad55e9aa904cae5bba512c315a8|ETH   |215.0             |2       |
|6125836 |0x013bbbf0154b9ad55e9aa904cae5bba512c315a8|ETH   |6.55183384        |1       |
|12246818|0x013bbbf0154b9ad55e9aa904cae5bba512c315a8|ETH   |0.7619261316546582|1       |
|2561356 |0x013bbbf0154b9ad55e9aa904cae5bba512c315a8|ETH   |5.84326448        |1       |
|5730173 |0x013bbbf0154b9ad55e9aa904cae5bba512c315a8|ETH   |1.0               |1       |
+--------+------------------------------------------+------+------------------+--------+
only showing top 5 rows



In [ ]:
df_baskets = df_portfolio.groupBy("user_id").agg(
    F.collect_set("symbol").alias("items")
)

df_baskets.show(10, truncate=False)

+--------+-----------------------------------+
|user_id |items                              |
+--------+-----------------------------------+
|10000370|[USDT, WETH, UCAP, GRG]            |
|10000594|[GT, WETH, LPT, LINK]              |
|10000626|[USDT, BADGER, LTO, G-CRE, YFED]   |
|10000657|[PRO, GRG, WBTC, FGP]              |
|10000783|[USDT]                             |
|10000868|[LDO, WETH, SWFTC, SNX]            |
|10001678|[WOO, OAX, REN]                    |
|10001800|[USDT, WETH]                       |
|10001885|[USDT, YFED]                       |
|10002032|[USDТ, TRAC, WETH, MANA, LINK, OMG]|
+--------+-----------------------------------+
only showing top 10 rows



In [ ]:
fpGrowth = FPGrowth(
    itemsCol="items",
    minSupport=0.001,
    minConfidence=0.5,
    numPartitions=100
)

# Training mo hinh tren tap du lieu da loc
model = fpGrowth.fit(df_baskets)

# Hien thi cac itemsets pho bien
model.freqItemsets.orderBy("freq", ascending=False).show(30, truncate=False)

+------------------+------+
|items             |freq  |
+------------------+------+
|[WETH]            |358275|
|[USDT]            |317972|
|[LINK]            |198871|
|[USDT, WETH]      |151714|
|[LINK, WETH]      |121713|
|[YFED]            |120593|
|[UCAP]            |115510|
|[UCAP, WETH]      |82514 |
|[YFED, USDT]      |73037 |
|[QNT]             |59052 |
|[USDC]            |58284 |
|[LINK, USDT]      |56163 |
|[YFED, WETH]      |46377 |
|[WBTC]            |42952 |
|[UCAP, USDT]      |35058 |
|[QNT, LINK]       |34858 |
|[USDC, WETH]      |33180 |
|[USDC, USDT]      |29450 |
|[UCAP, LINK]      |27635 |
|[WBTC, USDT]      |26218 |
|[LINK, USDT, WETH]|24963 |
|[GRG]             |24655 |
|[QNT, WETH]       |24173 |
|[USDТ]            |21531 |
|[USDС]            |20427 |
|[UCAP, USDT, WETH]|19999 |
|[WBTC, WETH]      |19933 |
|[LDO]             |19900 |
|[CRO]             |18309 |
|[USDC, LINK]      |17863 |
+------------------+------+
only showing top 30 rows



In [ ]:
model.associationRules.orderBy("confidence", ascending=False).show(20, truncate=False)

+------------------+----------+------------------+------------------+---------------------+
|antecedent        |consequent|confidence        |lift              |support              |
+------------------+----------+------------------+------------------+---------------------+
|[PRO, SNX, ANKR]  |[STORJ]   |0.9421593830334191|45.29634649656493 |0.0011991761185630172|
|[PRO, STORJ, ANKR]|[SNX]     |0.9361430395913155|38.03897104163467 |0.0011991761185630172|
|[PRO, STORJ, SNX] |[ANKR]    |0.9060568603213844|36.55643393016681 |0.0011991761185630172|
|[STORJ, SNX, ANKR]|[PRO]     |0.7932900432900433|39.63227779576369 |0.0011991761185630172|
|[UCAP]            |[WETH]    |0.7143450783481949|1.2187441830313843|0.1349915665035591   |
|[PRO, STORJ]      |[SNX]     |0.7121478873239436|28.93721548696553 |0.001323510886654135 |
|[PRO, SNX]        |[STORJ]   |0.7090271691498685|34.08801197297189 |0.001323510886654135 |
|[MDT, UCAP]       |[WETH]    |0.7048360200111173|1.2025207780053184|0.002074427

In [ ]:
df_rules_snowflake = model.associationRules.withColumn(
    "antecedent", F.concat_ws(",", F.col("antecedent"))
).withColumn(
    "consequent", F.concat_ws(",", F.col("consequent"))
).orderBy("confidence", ascending=False)

df_rules_snowflake.write.format(SNOWFLAKE_SOURCE_NAME) \
    .options(**sfOptionsStaging) \
    .option("dbtable", "FPGROWTH_RULES") \
    .mode("overwrite") \
    .save()

print(f"Da luu association rules vao bang Snowflake: {sfOptionsStaging['sfSchema']}.FPGROWTH_RULES")

Da luu association rules vao bang Snowflake: STAGING.FPGROWTH_RULES
